In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/My_Project/"
ZIP_FILE_PATH = os.path.join(DRIVE_PROJECT_PATH, "wisdm_data.zip")

print(f"Unzipping {ZIP_FILE_PATH} ... (if already unzipped this will overwrite nothing)")
!unzip -q "{ZIP_FILE_PATH}" -d "/content/temp_data/"

DATA_GLOB_PATTERN = "/content/temp_data/WISDM-51/data_*.csv"
print("Done. Looking for files at:", DATA_GLOB_PATTERN)


Unzipping /content/drive/MyDrive/My_Project/wisdm_data.zip ... (if already unzipped this will overwrite nothing)
Done. Looking for files at: /content/temp_data/WISDM-51/data_*.csv


In [ ]:

import pandas as pd
import numpy as np
import glob, os, math
print("Starting fast 24-feature extraction...")

WINDOW_SIZE = 60
STEP_SIZE = 30

file_list = glob.glob(DATA_GLOB_PATTERN)
if not file_list:
    raise FileNotFoundError("No files found - check DATA_GLOB_PATTERN or unzip step.")

def rms(a): return float(np.sqrt(np.mean(a*a)))
def energy(a): return float(np.sum(a*a))
def safe_skew(a, mean, std):
    if std == 0: return 0.0
    return float(np.mean((a-mean)**3) / (std**3))
def safe_kurt(a, mean, std):
    if std == 0: return 0.0
    return float(np.mean((a-mean)**4) / (std**4) - 3.0)

features = []
for file in file_list:
    try:
        df = pd.read_csv(file)
    except:
        print("Skipping unreadable:", file); continue
    df.columns = [c.strip() for c in df.columns]

    required_cols = ['X-accel','Y-accel','Z-accel','Activity Label','Subject-id']
    if not set(required_cols).issubset(set(df.columns)):
        print("Skipping (missing columns):", file); continue

    x_full = df['X-accel'].astype(float).to_numpy()
    y_full = df['Y-accel'].astype(float).to_numpy()
    z_full = df['Z-accel'].astype(float).to_numpy()
    acts = df['Activity Label'].to_numpy()
    subs = df['Subject-id'].to_numpy()

    n = len(df)
    for start in range(0, n - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE
        x = x_full[start:end]; y = y_full[start:end]; z = z_full[start:end]
        label = pd.Series(acts[start:end]).value_counts().idxmax()
        subject = pd.Series(subs[start:end]).value_counts().idxmax()

        xm, xs = float(x.mean()), float(x.std())
        ym, ys = float(y.mean()), float(y.std())
        zm, zs = float(z.mean()), float(z.std())

        row = {
            'subject': subject, 'activity': label,
            # X
            'x_mean': xm, 'x_std': xs, 'x_min': float(x.min()), 'x_max': float(x.max()),
            'x_rms': rms(x), 'x_energy': energy(x), 'x_skew': safe_skew(x, xm, xs), 'x_kurt': safe_kurt(x, xm, xs),
            # Y
            'y_mean': ym, 'y_std': ys, 'y_min': float(y.min()), 'y_max': float(y.max()),
            'y_rms': rms(y), 'y_energy': energy(y), 'y_skew': safe_skew(y, ym, ys), 'y_kurt': safe_kurt(y, ym, ys),
            # Z
            'z_mean': zm, 'z_std': zs, 'z_min': float(z.min()), 'z_max': float(z.max()),
            'z_rms': rms(z), 'z_energy': energy(z), 'z_skew': safe_skew(z, zm, zs), 'z_kurt': safe_kurt(z, zm, zs),
        }
        features.append(row)

features_df = pd.DataFrame(features)
out_path = os.path.join(DRIVE_PROJECT_PATH, "final_features.csv")
features_df.to_csv(out_path, index=False)
print("Feature extraction done. Windows:", len(features_df))
display(features_df.head())


Starting fast 24-feature extraction...
Feature extraction done. Windows: 160071


,subject,activity,x_mean,x_std,x_min,x_max,x_rms,x_energy,x_skew,x_kurt,...,y_skew,y_kurt,z_mean,z_std,z_min,z_max,z_rms,z_energy,z_skew,z_kurt
0,1620,A,3.817054,1.592910,0.485016,9.979980,4.136093,1026.435717,0.726043,2.422592,...,1.047445,2.702493,-1.851579,1.340869,-5.485672,0.908966,2.286104,313.576409,-0.323659,-0.308289
1,1620,A,3.283570,1.465180,0.178574,6.644165,3.595634,775.715036,0.076446,-0.442775,...,1.317183,2.475030,-1.349752,1.245172,-4.400238,1.275726,1.836378,202.336960,-0.366140,-0.523377
2,1620,A,3.249087,1.362062,0.178574,6.644165,3.523035,744.706696,0.203777,-0.081756,...,0.542850,0.316928,-1.472966,1.212615,-4.280060,1.275726,1.907895,218.403827,-0.194064,-0.607490
3,1620,A,3.223205,0.915691,1.191482,5.341690,3.350752,673.652306,0.172281,-0.180519,...,0.518345,0.534621,-1.514871,1.176805,-4.647675,0.506897,1.918255,220.782196,-0.722347,0.002955
4,1620,A,3.130116,1.125026,0.921295,6.227676,3.326155,663.798524,0.521620,0.517774,...,0.449509,0.183470,-1.593189,1.451960,-6.192505,0.862259,2.155560,278.786210,-0.715128,0.201054


In [ ]:


import os, glob
import numpy as np
import pandas as pd
from tqdm import tqdm

EXTRACT_DIR = "/content/temp_data/WISDM-51"
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/My_Project/"
OUT_NPZ = os.path.join(DRIVE_PROJECT_PATH, "raw_windows_fast.npz")

WINDOW_LEN = 60
STEP_LEN = 30

map_18_to_6 = {
    'A': 'walking',
    'B': 'jogging',
    'C': 'stairs',
    'D': 'sitting',
    'E': 'standing',
    'F': 'other','G': 'other','H': 'other','I': 'other','J': 'other',
    'K': 'other','L': 'other','M': 'other','O': 'other','P': 'other',
    'Q': 'other','R': 'other','S': 'other'
}
class_names = ['walking','jogging','stairs','sitting','standing','other']
class_to_idx = {c:i for i,c in enumerate(class_names)}


csv_files = glob.glob(os.path.join(EXTRACT_DIR, "data_*_accel_phone.csv"))
if not csv_files:
    csv_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "data_*_accel_phone.csv"), recursive=True)

print("Found", len(csv_files), "phone-accel CSV files")

X_list, y_list, sub_list = [], [], []

for fpath in tqdm(csv_files):
    try:
        df = pd.read_csv(fpath)
    except:
        continue

    df.columns = [c.strip() for c in df.columns]
    if not all(c in df.columns for c in ['Subject-id','Activity Label','X-accel','Y-accel','Z-accel']):
        continue


    x = df['X-accel'].to_numpy(dtype=np.float32)
    y = df['Y-accel'].to_numpy(dtype=np.float32)
    z = df['Z-accel'].to_numpy(dtype=np.float32)
    labels = df['Activity Label'].astype(str).to_numpy()
    subject = int(df['Subject-id'].iloc[0])

    N = len(df)
    i = 0
    while i + WINDOW_LEN <= N:

        xs = x[i:i+WINDOW_LEN]
        ys = y[i:i+WINDOW_LEN]
        zs = z[i:i+WINDOW_LEN]
        labs = labels[i:i+WINDOW_LEN]


        maj = pd.Series(labs).mode().iloc[0]
        if maj not in map_18_to_6:
            i += STEP_LEN
            continue

        macro = map_18_to_6[maj]
        macro_idx = class_to_idx[macro]

        window = np.vstack([xs,ys,zs]).T

        X_list.append(window)
        y_list.append(macro_idx)
        sub_list.append(subject)

        i += STEP_LEN

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
sub = np.array(sub_list, dtype=np.int32)

print("Created windows:", X.shape)
print("Class counts:", np.unique(y, return_counts=True))

np.savez_compressed(OUT_NPZ, X=X, y=y, sub=sub, classes=np.array(class_names))
print("Saved FAST windows to:", OUT_NPZ)



Found 51 phone-accel CSV files


100%|██████████| 51/51 [00:42<00:00,  1.20it/s]


Created windows: (160071, 60, 3)
Class counts: (array([0, 1, 2, 3, 4, 5]), array([  9311,   8942,   8526,   8815,   8982, 115495]))
Saved FAST windows to: /content/drive/MyDrive/My_Project/raw_windows_fast.npz


In [ ]:


import numpy as np
import os

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/My_Project/"
npz_path = os.path.join(DRIVE_PROJECT_PATH, "raw_windows_fast.npz")

data = np.load(npz_path, allow_pickle=True)

X_all = data["X"]
y_all = data["y"]
sub_all = data["sub"]
class_names = data["classes"]

print("Loaded:")
print("X:", X_all.shape, "y:", y_all.shape, "sub:", sub_all.shape)
print("Classes:", class_names)


Loaded:
X: (160071, 60, 3) y: (160071,) sub: (160071,)
Classes: ['walking' 'jogging' 'stairs' 'sitting' 'standing' 'other']


In [ ]:


from sklearn.model_selection import train_test_split

subjects = np.unique(sub_all)
print("Total subjects:", len(subjects))


train_subj, test_subj = train_test_split(
    subjects, test_size=0.25, random_state=42
)

mask_train = np.isin(sub_all, train_subj)
mask_test  = np.isin(sub_all, test_subj)

X_train = X_all[mask_train]
y_train = y_all[mask_train]
X_test  = X_all[mask_test]
y_test  = y_all[mask_test]

print("Train:", X_train.shape, "Test:", X_test.shape)


Total subjects: 51
Train: (123245, 60, 3) Test: (36826, 60, 3)


In [ ]:

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import pandas as pd


X_train_f = X_train.reshape(len(X_train), -1)
X_test_f  = X_test.reshape(len(X_test), -1)


cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw)}
print("Class weights:", cw_dict)

def evaluate(name, model):
    model.fit(X_train_f, y_train)
    y_pred = model.predict(X_test_f)

    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')

    print(f"\n=== {name} ===")
    print(f"Acc={acc:.4f}, BalancedAcc={bacc:.4f}, Kappa={kappa:.4f}, MacroF1={macro_f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    return (name, acc, bacc, kappa, macro_f1)

results = []


svm = LinearSVC(class_weight=cw_dict)
results.append(evaluate("Linear SVM", svm))


lr = LogisticRegression(max_iter=2000, class_weight=cw_dict)
results.append(evaluate("Logistic Regression", lr))


rf = RandomForestClassifier(n_estimators=150, class_weight=cw_dict)
results.append(evaluate("Random Forest", rf))

classical_df = pd.DataFrame(results, columns=["Model","Accuracy","BalancedAcc","Kappa","MacroF1"])
classical_df


Class weights: {0: np.float64(2.8788834384489603), 1: np.float64(3.0367878967080624), 2: np.float64(3.148020434227331), 3: np.float64(3.0171611829220524), 4: np.float64(2.9419698271746397), 5: np.float64(0.2307155185646947)}

=== Linear SVM ===
Acc=0.7212, BalancedAcc=0.1801, Kappa=0.0376, MacroF1=0.1648
              precision    recall  f1-score   support

     walking       0.07      0.00      0.01      2176
     jogging       0.62      0.08      0.14      2178
      stairs       0.04      0.00      0.00      2001
     sitting       0.00      0.00      0.00      2007
    standing       0.00      0.00      0.00      2000
       other       0.73      1.00      0.84     26464

    accuracy                           0.72     36826
   macro avg       0.24      0.18      0.16     36826
weighted avg       0.56      0.72      0.61     36826


=== Logistic Regression ===
Acc=0.1116, BalancedAcc=0.2460, Kappa=0.0335, MacroF1=0.1447
              precision    recall  f1-score   support

     w

,Model,Accuracy,BalancedAcc,Kappa,MacroF1
0,Linear SVM,0.721175,0.180067,0.037598,0.164798
1,Logistic Regression,0.111552,0.246045,0.033523,0.144662
2,Random Forest,0.770271,0.421805,0.377663,0.466360


In [ ]:
import tensorflow as tf
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available: 1


In [ ]:
tf.test.gpu_device_name()


'/device:GPU:0'

In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

num_classes = len(class_names)
input_shape = (60, 3)

def se_block(x, ratio=8):
    c = x.shape[-1]
    se = layers.GlobalAveragePooling1D()(x)
    se = layers.Dense(max(c//ratio,1), activation='relu')(se)
    se = layers.Dense(c, activation='sigmoid')(se)
    se = layers.Reshape((1,c))(se)
    return layers.Multiply()([x,se])

def build_model():
    inp = layers.Input(shape=input_shape)

    x = layers.Conv1D(64, 5, padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = se_block(x)

    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = se_block(x)

    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)

    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.LSTM(128)(x)

    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_model()
model.summary()


cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
cw_dict = {i: w for i, w in enumerate(cw)}

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=12,
    batch_size=256,
    class_weight=cw_dict,
    verbose=1
)


y_pred_probs = model.predict(X_test, batch_size=256)
y_pred = np.argmax(y_pred_probs, axis=1)

acc = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

print(f"\nDeepConvLSTM-SE FAST → Acc={acc:.4f}, BalancedAcc={bacc:.4f}, Kappa={kappa:.4f}, MacroF1={macro_f1:.4f}")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

dl_row = ("DeepConvLSTM-SE FAST", acc, bacc, kappa, macro_f1)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 60, 3)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 60, 64)    │      1,024 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 60, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 8)         │        520 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        576 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 64)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 60, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 60, 128)   │     24,704 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 128)   │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 16)        │      2,064 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │      2,176 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 1, 128)    │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_1          │ (None, 60, 128)   │          0 │ batch_normalizat… │
│ (Multiply)          │                   │            │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 60, 128)   │     49,280 │ multiply_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 60, 128)   │        512 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 60, 128)   │    131,584 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 128)       │    131,584 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 6)         │        774 │ dropout[0][0]   

 Total params: 345,566 (1.32 MB)

 Trainable params: 344,926 (1.32 MB)

 Non-trainable params: 640 (2.50 KB)

Epoch 1/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 452s 1s/step - accuracy: 0.4000 - loss: 0.8260 - val_accuracy: 0.4380 - val_loss: 1.2433
Epoch 2/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 427s 1s/step - accuracy: 0.5723 - loss: 0.4639 - val_accuracy: 0.4856 - val_loss: 1.2709
Epoch 3/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 445s 1s/step - accuracy: 0.6466 - loss: 0.3658 - val_accuracy: 0.4351 - val_loss: 1.4096
Epoch 4/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 430s 1s/step - accuracy: 0.6583 - loss: 0.3400 - val_accuracy: 0.6170 - val_loss: 1.1133
Epoch 5/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 399s 973ms/step - accuracy: 0.6859 - loss: 0.3056 - val_accuracy: 0.5437 - val_loss: 1.1646
Epoch 6/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 453s 1s/step - accuracy: 0.7357 - loss: 0.2758 - val_accuracy: 0.6020 - val_loss: 1.1713
Epoch 7/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 393s 959ms/step - accuracy: 0.7322 - loss: 0.2890 - val_accuracy: 0.5479 - val_loss: 1.2697
Epoch 8/12
410/410 ━━━━━━━━━━━━━━━━━━━━ 392s 957ms/step - accuracy: 0.7646 - loss: 0.2432 -